<a href="https://colab.research.google.com/github/rohinerd/document-qa-rag/blob/main/document_qa_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
print("hello")

hello


## 1. Install libraries

In [3]:
!pip install sentence-transformers chromadb mlflow -q

## 2. Load the guide text

In [4]:
!wget -q https://raw.githubusercontent.com/rohinerd/document-qa-rag/refs/heads/main/data/flexible-working.txt -O flexible-working.txt

In [5]:
with open("flexible-working.txt", encoding="utf-8") as f:
    text = f.read()

print(len(text), "characters")
print(text[:250])

7109 characters
Flexible working

1. Overview
Flexible working is a way of working that suits an employee’s needs, for example having flexible start and finish times, or working from home.

Flexible working rules are different in Northern Ireland.

All employees hav


## 3. Chunk the guide by section

In [6]:
def chunk_text(text, size=350, overlap=50):
    words = text.split()
    chunks = []
    step = size - overlap
    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + size])
        if chunk:
            chunks.append(chunk)
    return chunks

chunks = chunk_text(text)
print(len(chunks), "chunks")
print(chunks[0][:350])

4 chunks
Flexible working 1. Overview Flexible working is a way of working that suits an employee’s needs, for example having flexible start and finish times, or working from home. Flexible working rules are different in Northern Ireland. All employees have the legal right to request flexible working. Employees can request a change to: the number of hours t


In [7]:
print(len(text.split()), "words")
print(text[-300:])

1145 words
ng their request was treated as withdrawn
the date the employer should have responded to their request (but failed to do so)
If the problem happened before 1 October 2026, they may need to complain within 3 months.

If an employer or employee is unsure of their rights, they should get legal advice.



## 4. Build the Chroma index and retrieval

In [8]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
collection = client.get_or_create_collection("flexible_working")

embeddings = model.encode(chunks).tolist()
collection.add(
    ids=[f"chunk-{i}" for i in range(len(chunks))],
    embeddings=embeddings,
    documents=chunks,
)

def retrieve(question, k=3):
    query_embedding = model.encode([question]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=k)
    return results["documents"][0]

for q in [
    "Who has the legal right to request flexible working?",
    "From when can an employee make a request?",
    "Which aspects of work can an employee ask to change?",
]:
    print("Q:", q)
    for passage in retrieve(q):
        print("  -", passage[:120], "...")
    print()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Q: Who has the legal right to request flexible working?
  - Flexible working 1. Overview Flexible working is a way of working that suits an employee’s needs, for example having fle ...
  - limits) but works certain ‘core hours’, for example 10am to 4pm every day. Annualised hours The employee has to work a c ...
  - request a statement that this is a statutory request for flexible working details of how the employee wants to work flex ...

Q: From when can an employee make a request?
  - request a statement that this is a statutory request for flexible working details of how the employee wants to work flex ...
  - demand there’s a lack of work to do during the proposed working times the business is planning changes to the workforce  ...
  - Flexible working 1. Overview Flexible working is a way of working that suits an employee’s needs, for example having fle ...

Q: Which aspects of work can an employee ask to change?
  - limits) but works certain ‘core hours’, for example 10am to 4pm

In [9]:
def chunk_text(t, size=120, overlap=20):
    words = t.split()
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, len(words), step) if words[i:i + size]]

headings = ["Types of flexible working", "Applying for flexible working", "After the application", "Appeals"]

body_start = text.index("Flexible working is a way of working")
starts = [("Overview", body_start)]
pos = body_start
for name in headings:
    idx = text.index(name, pos)
    starts.append((name, idx))
    pos = idx + len(name)

bounds = starts + [("END", len(text))]
sections = {name: text[s:e] for (name, s), (_, e) in zip(bounds, bounds[1:])}

chunks, labels = [], []
for name, body in sections.items():
    for c in chunk_text(body):
        chunks.append(c)
        labels.append(name)

for name, body in sections.items():
    print(f"{name}: {len(body.split())} words")
print(len(chunks), "chunks")

Overview: 183 words
Types of flexible working: 223 words
Applying for flexible working: 293 words
After the application: 231 words
Appeals: 211 words
14 chunks


In [10]:
collection2 = client.get_or_create_collection("flexible_working_sections")
collection2.add(
    ids=[f"chunk-{i}" for i in range(len(chunks))],
    embeddings=model.encode(chunks).tolist(),
    documents=chunks,
    metadatas=[{"section": l} for l in labels],
)

def retrieve(question, k=3):
    res = collection2.query(query_embeddings=model.encode([question]).tolist(), n_results=k)
    return [(m["section"], d) for m, d in zip(res["metadatas"][0], res["documents"][0])]

## 5. Score retrieval against the eval set

In [11]:
import csv, io, urllib.request

EVAL_URL = "https://raw.githubusercontent.com/rohinerd/document-qa-rag/main/data/eval.csv"
rows = list(csv.DictReader(io.StringIO(urllib.request.urlopen(EVAL_URL).read().decode("utf-8"))))

hits1 = hits3 = 0
for row in rows:
    found = [s for s, _ in retrieve(row["question"], k=3)]
    hit1 = found[0] == row["source_section"]
    hit3 = row["source_section"] in found
    hits1 += hit1
    hits3 += hit3
    print(f'{row["id"]:>2}  top1={hit1}  top3={hit3}  expected={row["source_section"]}  got={found}')

print(f"\nTop-1 section hit rate: {hits1}/{len(rows)}")
print(f"Top-3 section hit rate: {hits3}/{len(rows)}")

 1  top1=True  top3=True  expected=Overview  got=['Overview', 'Applying for flexible working', 'Applying for flexible working']
 2  top1=False  top3=True  expected=Overview  got=['After the application', 'Overview', 'Appeals']
 3  top1=False  top3=False  expected=Overview  got=['Applying for flexible working', 'After the application', 'Applying for flexible working']
 4  top1=True  top3=True  expected=Overview  got=['Overview', 'Applying for flexible working', 'After the application']
 5  top1=True  top3=True  expected=Overview  got=['Overview', 'Overview', 'Applying for flexible working']
 6  top1=False  top3=False  expected=Overview  got=['Appeals', 'Appeals', 'Applying for flexible working']
 7  top1=True  top3=True  expected=Overview  got=['Overview', 'Appeals', 'After the application']
 8  top1=True  top3=True  expected=Overview  got=['Overview', 'Overview', 'After the application']
 9  top1=True  top3=True  expected=Appeals  got=['Appeals', 'After the application', 'After the app

In [12]:
import mlflow

def build_and_score(size, overlap):
    all_chunks, all_labels = [], []
    for name, body in sections.items():
        for c in chunk_text(body, size=size, overlap=overlap):
            all_chunks.append(c)
            all_labels.append(name)

    coll = client.get_or_create_collection(f"fw_v2_{size}_{overlap}")
    coll.add(
        ids=[f"chunk-{i}" for i in range(len(all_chunks))],
        embeddings=model.encode(all_chunks).tolist(),
        documents=all_chunks,
        metadatas=[{"section": l} for l in all_labels],
    )

    h1 = h3 = 0
    for row in rows:
        res = coll.query(
            query_embeddings=model.encode([row["question"]]).tolist(),
            n_results=3,
        )
        found = [m["section"] for m in res["metadatas"][0]]
        h1 += found[0] == row["source_section"]
        h3 += row["source_section"] in found
    return len(all_chunks), h1 / len(rows), h3 / len(rows)

mlflow.set_experiment("chunk-size-comparison")
for size, overlap in [(120, 20), (200, 40), (350, 50)]:
    n, t1, t3 = build_and_score(size, overlap)
    with mlflow.start_run(run_name=f"size{size}_overlap{overlap}"):
        mlflow.log_params({"chunk_size": size, "overlap": overlap, "k": 3, "num_chunks": n})
        mlflow.log_metrics({"top1_hit_rate": t1, "top3_hit_rate": t3})
    print(f"size={size} overlap={overlap} chunks={n} top1={t1:.2f} top3={t3:.2f}")

2026/11/08 22:16:34 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/08 22:16:34 INFO mlflow.store.db.utils: Updating database tables
2026/11/08 22:16:40 INFO mlflow.tracking.fluent: Experiment with name 'chunk-size-comparison' does not exist. Creating a new experiment.


size=120 overlap=20 chunks=14 top1=0.68 top3=0.89
size=200 overlap=40 chunks=10 top1=0.58 top3=0.89
size=350 overlap=50 chunks=5 top1=0.63 top3=0.95


In [13]:
print(sections["Types of flexible working"])
print("-----")
print(sections["Applying for flexible working"])

Types of flexible working
There are different ways of working flexibly.

Job sharing
Two people do one job and split the hours.

Remote working and working from home
Working from anywhere other than the employee’s usual workplace. It can include working from home.

Hybrid working
A combination of working remotely and working in the employee’s usual workplace.

Part time
Working less than full-time hours (usually by working fewer days).

Compressed hours
Working full-time hours but over fewer days - for example a 9-day fortnight (an employee has a day off every other week and works longer hours on the other days)

Flexitime
The employee chooses when to start and end work (within agreed limits) but works certain ‘core hours’, for example 10am to 4pm every day.

Annualised hours
The employee has to work a certain number of hours over the year but they have some flexibility about when they work. There are sometimes ‘core hours’ which the employee regularly works each week, and they work th

In [14]:
!ls

flexible-working.txt  mlflow.db  sample_data


## 6. Compare MRR and NDCG

In [15]:
import math

def reciprocal_rank(found, expected):
    for i, s in enumerate(found, 1):
        if s == expected:
            return 1 / i
    return 0.0

def ndcg_at_k(found, expected, k=3):
    # One relevant section per question, so ideal DCG is 1
    for i, s in enumerate(found[:k], 1):
        if s == expected:
            return 1 / math.log2(i + 1)
    return 0.0

mrr_scores, ndcg_scores = [], []
for row in rows:
    found = [s for s, _ in retrieve(row["question"], k=3)]
    mrr_scores.append(reciprocal_rank(found, row["source_section"]))
    ndcg_scores.append(ndcg_at_k(found, row["source_section"]))

print(f"MRR:    {sum(mrr_scores)/len(rows):.3f}")
print(f"NDCG@3: {sum(ndcg_scores)/len(rows):.3f}")

MRR:    0.789
NDCG@3: 0.817


## 7. Log MRR and NDCG to MLflow

Logs the 120-word run's MRR and NDCG@3 so they appear next to the chunk-size comparison in MLflow.

In [16]:
import mlflow

mlflow.set_experiment("chunk-size-comparison")
with mlflow.start_run(run_name="size120_overlap20_mrr_ndcg"):
    mlflow.log_params({"chunk_size": 120, "overlap": 20, "k": 3, "num_questions": len(rows)})
    mlflow.log_metrics({
        "mrr": sum(mrr_scores) / len(rows),
        "ndcg_at_3": sum(ndcg_scores) / len(rows),
    })
print("Logged MRR and NDCG to MLflow")

Logged MRR and NDCG to MLflow


## 8. Compare all chunk sizes with MRR and NDCG

Rebuilds the index at three chunk sizes and logs top-1, top-3, MRR, and NDCG@3 for each run in MLflow.

In [17]:
import math
import mlflow

def reciprocal_rank(found, expected):
    for i, s in enumerate(found, 1):
        if s == expected:
            return 1 / i
    return 0.0

def ndcg_at_k(found, expected, k=3):
    for i, s in enumerate(found[:k], 1):
        if s == expected:
            return 1 / math.log2(i + 1)
    return 0.0

def build_and_score(size, overlap):
    all_chunks, all_labels = [], []
    for name, body in sections.items():
        for c in chunk_text(body, size=size, overlap=overlap):
            all_chunks.append(c)
            all_labels.append(name)

    coll = client.get_or_create_collection(f"fw_v3_{size}_{overlap}")
    coll.add(
        ids=[f"chunk-{i}" for i in range(len(all_chunks))],
        embeddings=model.encode(all_chunks).tolist(),
        documents=all_chunks,
        metadatas=[{"section": l} for l in all_labels],
    )

    h1 = h3 = 0
    mrr, ndcg = [], []
    for row in rows:
        res = coll.query(
            query_embeddings=model.encode([row["question"]]).tolist(),
            n_results=3,
        )
        found = [m["section"] for m in res["metadatas"][0]]
        h1 += found[0] == row["source_section"]
        h3 += row["source_section"] in found
        mrr.append(reciprocal_rank(found, row["source_section"]))
        ndcg.append(ndcg_at_k(found, row["source_section"]))

    n = len(rows)
    return {
        "chunks": len(all_chunks),
        "top1": h1 / n,
        "top3": h3 / n,
        "mrr": sum(mrr) / n,
        "ndcg_at_3": sum(ndcg) / n,
    }

mlflow.set_experiment("chunk-size-comparison")
for size, overlap in [(120, 20), (200, 40), (350, 50)]:
    r = build_and_score(size, overlap)
    with mlflow.start_run(run_name=f"size{size}_overlap{overlap}_v3"):
        mlflow.log_params({"chunk_size": size, "overlap": overlap, "k": 3, "num_chunks": r["chunks"], "num_questions": len(rows)})
        mlflow.log_metrics({k: v for k, v in r.items() if k != "chunks"})
    print(f"size={size} overlap={overlap} chunks={r['chunks']} top1={r['top1']:.2f} top3={r['top3']:.2f} mrr={r['mrr']:.3f} ndcg={r['ndcg_at_3']:.3f}")

size=120 overlap=20 chunks=14 top1=0.68 top3=0.89 mrr=0.789 ndcg=0.817
size=200 overlap=40 chunks=10 top1=0.58 top3=0.89 mrr=0.728 ndcg=0.771
size=350 overlap=50 chunks=5 top1=0.63 top3=0.95 mrr=0.763 ndcg=0.810


## 9. Connect to the language model (Gemini)

Loads the Google AI Studio API key from Colab Secrets. The key is never printed or saved in the notebook.

In [18]:
!pip install google-genai -q

from google import genai
from google.colab import userdata

client_llm = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
print("Key loaded")

Key loaded


### Check which models the API key can use

In [19]:
for m in client_llm.models.list():
    if "flash" in m.name and "generateContent" in (m.supported_actions or []):
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts


## 10. Generate a cited answer from the retrieved passages

## 10a. Define prompt versions

In [20]:
PROMPT_V1 = """Answer the question using only the passages below.
Cite the section name in square brackets after each claim, for example [Overview].
If the passages don't contain the answer, say "The guide doesn't answer this."

Passages:
{context}

Question: {question}
Answer:"""

In [21]:
PROMPT_V2 = """You are answering questions about UK flexible working rules using only the passages provided.
Rules:
1. Use only facts stated in the passages.
2. Put the section name in square brackets after every claim.
3. If the passages do not contain the answer, reply exactly: The guide doesn't answer this.
4. Give a complete list when the question asks for one.

Passages:
{context}

Question: {question}
Answer:"""

In [38]:
PROMPT_V3 = """You are answering questions about UK flexible working rules using only the passages provided.

Rules:
1. Use only facts stated in the passages.
2. After each claim, cite the section name in square brackets, using the plain name only, such as [Overview] or [Appeals]. The passages have labels like "[Section: Overview]". Those labels are metadata. Never copy the word "Section:" into your citation.
3. Copy section names exactly as they are written in the labels, with correct spelling.
4. If the passages do not contain the answer, reply exactly: The guide doesn't answer this.
5. When the question asks for a list, give the complete list.

Example of the required format:
Question: What is the minimum notice period for a flexible working request?
Answer: The example passages do not state a notice period. [Overview]

Passages:
{context}

Question: {question}
Answer:"""

In [22]:
MODEL = "gemini-flash-lite-latest"

def answer(question, template, k=3):
    hits = retrieve(question, k=k)
    context = "\n\n".join(f"[Section: {sec}]\n{passage}" for sec, passage in hits)
    prompt = template.format(context=context, question=question)
    response = client_llm.models.generate_content(model=MODEL, contents=prompt)
    return response.text

for q in [
    "From when can an employee make a request?",
    "How long does an employer have to decide on a request?",
    "What is the deadline for a tribunal complaint?",
]:
    print("Q:", q)
    try:
        print("A (v1):", answer(q, PROMPT_V1))
        print("A (v2):", answer(q, PROMPT_V2))
    except Exception as e:
        print("Error retrieving answer:", e)
    print()

Q: From when can an employee make a request?
A (v1): The guide doesn't answer this.
A (v2): The guide doesn't answer this.

Q: How long does an employer have to decide on a request?
A (v1): An employer must make a decision on a flexible working request within 2 months of the request (or longer if agreed with the employee) [After the application].
A (v2): An employer has 2 months from the request (or longer if agreed with the employee) to make a decision [After the application].

Q: What is the deadline for a tribunal complaint?
A (v1): An employee should usually complain to the tribunal within 6 months of:
* hearing their employer’s decision on their application or appeal
* hearing their request was treated as withdrawn
* the date the employer should have responded to their request (but failed to do so)

If the problem happened before 1 October 2026, they may need to complain within 3 months [Appeals].
A (v2): An employee should usually complain to the tribunal within 6 months of:
- he

## 11. Score the answer step (manual check against the guide)

In [29]:
import mlflow

answer_results = [
    {"question": "From when can an employee make a request?",
     "answer_correct": False, "citation_correct": False,
     "note": "Refused; retrieval missed the Overview passage"},
    {"question": "How long does an employer have to decide on a request?",
     "answer_correct": True, "citation_correct": False,
     "note": "Correct two-month deadline; cited After the application instead of Applying"},
    {"question": "What is the deadline for a tribunal complaint?",
     "answer_correct": True, "citation_correct": True,
     "note": "Correct, including the 3-month exception; cited Appeals"},
]

n = len(answer_results)
answer_acc = sum(r["answer_correct"] for r in answer_results) / n
citation_acc = sum(r["citation_correct"] for r in answer_results) / n

mlflow.set_experiment("answer-step-test")
with mlflow.start_run(run_name="gemini_answer_3q"):
    mlflow.log_params({"model": MODEL, "k": 3, "num_questions": n, "judge": "manual"})
    mlflow.log_metrics({"answer_accuracy": answer_acc, "citation_accuracy": citation_acc})

print(f"Answer accuracy: {answer_acc:.2f}  Citation accuracy: {citation_acc:.2f}")

Answer accuracy: 0.67  Citation accuracy: 0.33


## 12. Generate answers for all 19 evaluation questions

In [32]:
import time
import csv

results_llm = []
for row in rows:
    try:
        a = answer(row["question"], PROMPT_V1)
    except Exception as e:
        a = f"ERROR: {e}"
    results_llm.append({
        "id": row["id"],
        "question": row["question"],
        "expected_section": row["source_section"],
        "answer": a,
    })
    print(f'[{row["id"]}] Q: {row["question"]}\nA: {a}\n')
    time.sleep(5)

with open("answers_19q.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "question", "expected_section", "answer"])
    writer.writeheader()
    writer.writerows(results_llm)
print("Saved answers_19q.csv")

[1] Q: Who has the legal right to request flexible working?
A: All employees have the legal right to request flexible working [Overview].

[2] Q: Which aspects of work can an employee ask to change?
A: An employee can ask to change:
* The number of hours they work [Overview].
* When they start or finish work [Overview].
* The days they work [Overview].
* Where they work [Overview].

[3] Q: From when can an employee make a request?
A: The provided passages do not state the date or time from when an employee can make a request, only that employees can make 2 applications for flexible working in any 12-month period [Applying for flexible working].

[4] Q: What must an employer do when handling a request?
A: When handling a request, an employer must:

* Consider flexible working requests in a 'reasonable manner', which includes examples such as assessing the advantages and disadvantages of the application, discussing possible alternatives to the request, and offering an appeal process [Ove

## 12a. Score prompt versions 1 and 2 on the three-question test

In [33]:
import mlflow

mlflow.set_experiment("prompt-versions")

scores_prompt = {
    "v1": {"answer_correct": [False, True, True], "citation_correct": [False, False, True]},
    "v2": {"answer_correct": [False, True, True], "citation_correct": [False, False, False]},
}

for name, s in scores_prompt.items():
    n = len(s["answer_correct"])
    with mlflow.start_run(run_name=f"prompt_{name}_3q"):
        mlflow.log_params({"prompt_version": name, "model": MODEL, "k": 3, "num_questions": n, "judge": "manual"})
        mlflow.log_metrics({
            "answer_accuracy": sum(s["answer_correct"]) / n,
            "citation_accuracy": sum(s["citation_correct"]) / n,
        })
    print(name, "answer:", round(sum(s["answer_correct"]) / n, 2), "citation:", round(sum(s["citation_correct"]) / n, 2))

v1 answer: 0.67 citation: 0.33
v2 answer: 0.67 citation: 0.0


## 12b. Log prompt versions 1, 2, and 3 to MLflow

In [41]:
import mlflow

scores_all = {
    "v1": {"answer": 16, "citation": 15},
    "v2": {"answer": 16, "citation": 15},
    "v3": {"answer": 16, "citation": 17},
}

n = 19
mlflow.set_experiment("prompt-versions")

for version, s in scores_all.items():
    with mlflow.start_run(run_name=f"prompt_{version}_19q"):
        mlflow.log_params({
            "prompt_version": version,
            "model": MODEL,
            "k": 3,
            "num_questions": n,
            "judge": "manual",
        })
        mlflow.log_metrics({
            "answer_accuracy": s["answer"] / n,
            "citation_accuracy": s["citation"] / n,
        })
    print(f"{version}: answer {s['answer']}/{n}, citation {s['citation']}/{n}")

v1: answer 16/19, citation 15/19
v2: answer 16/19, citation 15/19
v3: answer 16/19, citation 17/19


## 13. Run prompt version 2 on all 19 questions

In [40]:
import time
import csv

results_v2 = []
for row in rows:
    try:
        a = answer(row["question"], PROMPT_V3)
    except Exception as e:
        a = f"ERROR: {e}"
    results_v2.append({
        "id": row["id"],
        "question": row["question"],
        "expected_section": row["source_section"],
        "answer": a,
    })
    print(f'[{row["id"]}] Q: {row["question"]}\nA: {a}\n')
    time.sleep(5)

with open("answers_19q_v3.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "question", "expected_section", "answer"])
    writer.writeheader()
    writer.writerows(results_v2)
print("Saved answers_19q_v3.csv")

[1] Q: Who has the legal right to request flexible working?
A: All employees have the legal right to request flexible working. [Overview]

[2] Q: Which aspects of work can an employee ask to change?
A: An employee can ask to change:
- the number of hours they work [Overview]
- when they start or finish work [Overview]
- the days they work [Overview]
- where they work [Overview]

[3] Q: From when can an employee make a request?
A: The guide doesn't answer this.

[4] Q: What must an employer do when handling a request?
A: When handling a request, an employer must:
- Assess the advantages and disadvantages of the application [Overview]
- Discuss possible alternatives to the request [Overview]
- Offer an appeal process [Overview]
- Discuss the employee’s request with them before refusing their application [Applying for flexible working]
- Make a decision within 2 months of the request (or longer if agreed with the employee) [After the application]

[5] Q: Examples of reasonable handling in

## 14. Score all 19 answers and log to MLflow

In [ ]:
import mlflow

scores = {
    1: (True, True), 2: (True, True), 3: (False, False), 4: (True, True),
    5: (True, True), 6: (False, False), 7: (True, True), 8: (True, True),
    9: (True, True), 10: (True, True), 11: (True, True), 12: (True, True),
    13: (True, True), 14: (True, True), 15: (True, True), 16: (True, True),
    17: (True, True), 18: (False, True), 19: (True, True),
}

n = len(scores)
answer_acc = sum(a for a, _ in scores.values()) / n
citation_acc = sum(c for _, c in scores.values()) / n

mlflow.set_experiment("answer-step-test")
with mlflow.start_run(run_name="gemini_answer_19q"):
    mlflow.log_params({"model": MODEL, "k": 3, "num_questions": n, "judge": "manual"})
    mlflow.log_metrics({"answer_accuracy": answer_acc, "citation_accuracy": citation_acc})

print(f"19q  Answer accuracy: {answer_acc:.2f}  Citation accuracy: {citation_acc:.2f}")

19q  Answer accuracy: 0.84  Citation accuracy: 0.89
